# Mirror Auto-Align — Interactive Notebook

A friendly control panel for the BL 3.3.2 order-sorter simulation. You don't have to write any Python — just run each cell and use the sliders.

**How to run a cell:** click on it, then press **Shift + Enter**. The first time, go top to bottom in order.


## One-time setup

1. Install **Anaconda** (the simplest way to get Python + Jupyter on Windows): https://www.anaconda.com/download . After it installs, open **Jupyter Notebook** from the Start menu — it opens in your web browser.
2. Make sure this notebook sits in the **same folder** as the simulation files (`physics.py`, `actuator.py`, `detector.py`, `controller.py`, `run_sim.py`, `sample.py`, `acquisition.py`). It already does if you opened it from the `mirror_attenuation_sim` folder.
3. Run the cell just below **once** to add the two extra packages the sliders need.


In [1]:
# Run this ONCE. It adds the two packages that don't come bundled with Anaconda.
# Safe to re-run — it will just say "already satisfied".
%pip install xraydb ipywidgets --quiet
print("Setup done. If ipywidgets was just installed, do Kernel > Restart, then run the cells below.")

Note: you may need to restart the kernel to use updated packages.
Setup done. If ipywidgets was just installed, do Kernel > Restart, then run the cells below.


In [2]:
# Load the simulation. IMPORTANT: this notebook must be in the same folder as the .py files.
import physics, controller, detector, run_sim, sample, acquisition
import ipywidgets as widgets
from IPython.display import Image, display

print("All modules loaded — you're ready to go.")

All modules loaded — you're ready to go.


## 1) Align to a target pileup

Move the sliders, then click the **Run interact** button. The loop drives the (blind, non-repeatable) PIM05 mirror until the detector sees the pileup you asked for. The left plot shows it converging; the middle shows the true angle it found.


In [3]:
def align(energy_keV, pileup_percent, frames, frame_length_s):
    h = controller.auto_align(E=energy_keV, p_target=pileup_percent/100.0,
                              n_frames=frames, frame_s=frame_length_s,
                              seed=1, verbose=False)
    controller.plot(h, path="controller_sim.png")
    display(Image("controller_sim.png"))
    print(f"Converged to {h['p_meas'][-1]*100:.2f}% pileup "
          f"(true angle {h['theta'][-1]:.4f} deg, {h['net_steps']:+d} net steps).")

widgets.interact_manual(align,
    energy_keV=widgets.FloatSlider(min=4, max=10, step=0.5, value=8.0, description="Energy keV"),
    pileup_percent=widgets.FloatSlider(min=1, max=10, step=0.5, value=1.0, description="Pileup %"),
    frames=widgets.IntSlider(min=5, max=100, step=5, value=20, description="Frames"),
    frame_length_s=widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.2, description="Frame s"));

interactive(children=(FloatSlider(value=8.0, description='Energy keV', max=10.0, min=4.0, step=0.5), FloatSlid…

## 2) Energy sweep vs the operating recipe

Runs the blind loop at several energies (4–10 keV) and overlays the angles it finds on the deterministic recipe curve. Takes a few seconds.


In [4]:
def sweep(pileup_percent):
    data = run_sim.energy_sweep(energies=(4,5,6,7,8,9,10), p_target=pileup_percent/100.0)
    run_sim.plot(data, p_target=pileup_percent/100.0, path="run_sim.png")
    display(Image("run_sim.png"))

widgets.interact_manual(sweep,
    pileup_percent=widgets.FloatSlider(min=1, max=10, step=0.5, value=1.0, description="Pileup %"));

interactive(children=(FloatSlider(value=1.0, description='Pileup %', max=10.0, min=1.0, step=0.5), Button(desc…

## 3) Simulate one detector acquisition (real MYTHEN2 format)

Produces a run of frames with realistic Poisson noise and pileup, and plots the summed spectrum plus the per-frame signal. Tick **Save real files** to also write the real `Acquisition/Frame####.dat` + `.cfg` files into a folder you can open and analyze like real data.


In [5]:
def acquire(energy_keV, pileup_percent, frames, frame_length_s, save_real_files):
    det = detector.MythenDetector(energy_keV=energy_keV, seed=0)
    strip_target = physics.true_rate_for_pileup(pileup_percent/100.0, energy_keV)
    det.acquire(beam_rate=strip_target/det.peak_frac, n_frames=frames, frame_s=frame_length_s)
    det.plot(path="detector_sim.png")
    display(Image("detector_sim.png"))
    print(f"Pileup measured from the data: {det.measured_pileup()*100:.2f}%")
    if save_real_files:
        folder = det.write_acquisition(out_dir=".", acq_number=999)
        print(f"Wrote real-format files to: {folder}")

widgets.interact_manual(acquire,
    energy_keV=widgets.FloatSlider(min=4, max=10, step=0.5, value=8.0, description="Energy keV"),
    pileup_percent=widgets.FloatSlider(min=1, max=10, step=0.5, value=1.0, description="Pileup %"),
    frames=widgets.IntSlider(min=5, max=200, step=5, value=100, description="Frames"),
    frame_length_s=widgets.FloatSlider(min=0.05, max=2.0, step=0.05, value=1.0, description="Frame s"),
    save_real_files=widgets.Checkbox(value=False, description="Save real files"));

interactive(children=(FloatSlider(value=8.0, description='Energy keV', max=10.0, min=4.0, step=0.5), FloatSlid…

## 4) Convergence budget: cold vs warm start

Compares how long and how many motor steps alignment takes, cold-starting every time vs warm-starting from the previous point. This is the slowest cell (about a minute at 4 seeds).


In [6]:
def run_budget(seeds):
    cold = run_sim.budget_sweep(p_target=0.01, warm_start=False, n_seeds=seeds)
    warm = run_sim.budget_sweep(p_target=0.01, warm_start=True,  n_seeds=seeds)
    run_sim.plot_budget(cold, warm, path="run_sim_budget.png")
    display(Image("run_sim_budget.png"))

widgets.interact_manual(run_budget,
    seeds=widgets.IntSlider(min=1, max=8, step=1, value=4, description="Seeds"));

interactive(children=(IntSlider(value=4, description='Seeds', max=8, min=1), Button(description='Run Interact'…

## 5) Full automated acquisition run (the sample scan)

This runs the whole recipe end to end, exactly as it would on the beamline:

For each energy in your scan: move the **sample out** (rotate flat + lift 2 mm), **auto-align the mirror** to your pileup target on the direct beam, move the **sample back in**, then **rotate through your grazing angles** and record a spectrum at each. You get one figure of spectra per energy, a final **rocking curve** (integrated intensity vs grazing angle), and a printed log of every move.

**You choose the scan:** the energy range (**E min / E max**) and how many energies (**E points**), and the angle range (**Angle min / max**) and how many angles (**Angle points**). Both endpoints are included. More points = finer scan but longer run.

The sample is a 200 ℓ/mm bare-Si grating; each spectrum dims as the grazing angle passes the critical angle. The **Slit** slider sets beam height (wider = brighter but broader peaks); **Pileup %** sets the calibrated flux. High-angle spectra are faint, so a longer frame (0.5–2 s) helps.

Tick **Save real files** to also write every spectrum in the real MYTHEN `Acquisition/Frame####.dat` + `.cfg` format into a `scan_acquisitions` folder.


In [7]:
def full_run(E_min, E_max, E_points, angle_min, angle_max, angle_points,
             pileup_percent, slit_um, frames, frame_length_s, save_real_files, show_log):
    paths, log = acquisition.run_full_sequence(
        E_min=E_min, E_max=E_max, n_energies=E_points,
        alpha_min=angle_min, alpha_max=angle_max, n_angles=angle_points,
        p_target=pileup_percent/100.0, slit_um=slit_um,
        n_frames=frames, frame_s=frame_length_s,
        save_files=save_real_files, verbose=False)
    if show_log:
        for line in log:
            print(line)
    if save_real_files:
        print(f"Saved {E_points*angle_points} real-format acquisitions to scan_acquisitions/.")
    for p in paths:          # one figure per energy, then the rocking-curve summary
        display(Image(p))

widgets.interact_manual(full_run,
    E_min=widgets.FloatSlider(min=4, max=10, step=0.5, value=4.0, description="E min keV"),
    E_max=widgets.FloatSlider(min=4, max=10, step=0.5, value=10.0, description="E max keV"),
    E_points=widgets.IntSlider(min=2, max=10, step=1, value=7, description="E points"),
    angle_min=widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=0.2, description="Angle min"),
    angle_max=widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=0.7, description="Angle max"),
    angle_points=widgets.IntSlider(min=2, max=9, step=1, value=6, description="Angle points"),
    pileup_percent=widgets.FloatSlider(min=1, max=10, step=0.5, value=1.0, description="Pileup %"),
    slit_um=widgets.IntSlider(min=20, max=200, step=10, value=100, description="Slit um"),
    frames=widgets.IntSlider(min=5, max=100, step=5, value=20, description="Frames"),
    frame_length_s=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description="Frame s"),
    save_real_files=widgets.Checkbox(value=False, description="Save real files"),
    show_log=widgets.Checkbox(value=True, description="Show step log"));

interactive(children=(FloatSlider(value=4.0, description='E min keV', max=10.0, min=4.0, step=0.5), FloatSlide…

## 6) Slit-height explorer

The vertical slit sets the beam height, and this panel lets you feel the tradeoff. Pick an energy, a sample grazing angle, a slit height, and a pileup %, and see **(left)** the spectrum you'd record and **(right)** how the beam size on the detector depends on slit height.

Two reference marks on the right plot: the **Nyquist target** (beam = 2 strips = 100 µm — Howard's rule, so a peak lands on ~2 strips and you can find its centre by interpolation), and the **diffraction optimum** (the smallest beam achievable — shrinking the slit past this makes the spot *bigger*, not sharper). A wider slit gives more flux but broader peaks; a narrower slit sharpens peaks until diffraction takes over, and costs flux.

The **Pileup %** slider sets the mirror calibration on the direct beam — it scales the overall brightness and is independent of the grazing angle.


In [8]:
def slit_explore(energy_keV, grazing_deg, slit_um, pileup_percent):
    path, info = acquisition.slit_demo(E=energy_keV, alpha=grazing_deg, slit_um=slit_um,
                                       p_target=pileup_percent/100.0)
    display(Image(path))
    print(f"beam at detector: {info['beam_fwhm_um']:.0f} um = {info['beam_strips']:.1f} strips "
          f"(Nyquist wants ~2).")
    print(f"delivered flux {info['delivered_flux']:.2e} ph/s; "
          f"diffraction-limited slit ~{info['optimal_slit_um']:.0f} um.")

widgets.interact_manual(slit_explore,
    energy_keV=widgets.FloatSlider(min=4, max=10, step=0.5, value=8.0, description="Energy keV"),
    grazing_deg=widgets.FloatSlider(min=0.2, max=0.7, step=0.1, value=0.3, description="Grazing deg"),
    slit_um=widgets.IntSlider(min=15, max=200, step=5, value=100, description="Slit um"),
    pileup_percent=widgets.FloatSlider(min=1, max=10, step=0.5, value=1.0, description="Pileup %"));

interactive(children=(FloatSlider(value=8.0, description='Energy keV', max=10.0, min=4.0, step=0.5), FloatSlid…

## If something goes wrong

- **"No module named physics"** — the notebook isn't in the same folder as the `.py` files. Move it into `mirror_attenuation_sim` and reopen it from there.
- **No sliders or no button appears** — run the setup cell, then do **Kernel > Restart** and run the cells again.
- **"object has no attribute ..." after I changed code** — Jupyter is holding an old copy in memory. Do **Kernel > Restart & Run All**.
- **A plot looks stale** — click **Run interact** again; each run overwrites the image.
- Nothing here touches real hardware — it's all simulation, so poke at it freely.
